# EDA and Machine Learning — вступительное задание ИТМО «Искусственный интеллект»

Задача: бинарная классификация `relief_granted` (компания закрыла обращение с денежной или неденежной
компенсацией) для потока потребительских жалоб — фича «Приоритет обращения».

**Данные:** `data/complaints_train.csv` (~7.7 ГБ), `data/complaints_test.csv`, `data/sample_submission.csv`.
Отложенная часть — последние месяцы наблюдений, то есть будущее относительно обучающей.

**Инструменты:** DuckDB для операций по полному датасету (обучающий CSV не загружается целиком в pandas),
pandas / matplotlib для уже уменьшенных до безопасного размера выборок и визуализаций.

## Setup — общее окружение для всех заданий

Ячейка ниже выполняется один раз и определяет пути к данным, соединение DuckDB и SQL-выражения
для чтения исходных CSV. Все последующие задания переиспользуют их.

Важное замечание о чтении данных: текстовое поле `Consumer.Complaint.Narrative` содержит переносы строк
внутри закавыченных значений, поэтому файл читается как настоящий RFC-4180 CSV. Параллельный CSV-ридер
DuckDB на этом файле не работает (`Parallel CSV Reader currently does not support a full read on this file`),
поэтому используется `parallel=false`. Типы читаются как строки (`all_varchar=true`): данные сырые
(несколько форматов дат, мусорные значения), автоматическому выводу типов доверять нельзя — поля
разбираются осознанно там, где это нужно по заданию.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    """Ноутбук должен работать и из notebooks/, и из корня репозитория."""
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Не найден корень репозитория с каталогом data/")


ROOT = _find_repo_root(Path.cwd().resolve())
DATA = ROOT / "data"
TRAIN_CSV = DATA / "complaints_train.csv"
TEST_CSV = DATA / "complaints_test.csv"
SAMPLE_SUB_CSV = DATA / "sample_submission.csv"

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")  # чтобы виджет прогресса не попадал в вывод


def csv_src(path: Path) -> str:
    """SQL-выражение чтения сырого CSV: строгий разбор кавычек, без вывода типов."""
    return (
        f"read_csv('{path.as_posix()}', header=true, all_varchar=true, parallel=false)"
    )


TRAIN = csv_src(TRAIN_CSV)
TEST = csv_src(TEST_CSV)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

print(f"ROOT = {ROOT}")
for p in (TRAIN_CSV, TEST_CSV, SAMPLE_SUB_CSV):
    print(f"{p.name:>26}  exists={p.is_file()}  size={p.stat().st_size / 2**30:.2f} GiB")
print(f"duckdb {duckdb.__version__}, pandas {pd.__version__}")

ROOT = C:\work\eda-ml
      complaints_train.csv  exists=True  size=7.19 GiB
       complaints_test.csv  exists=True  size=0.39 GiB
     sample_submission.csv  exists=True  size=0.03 GiB
duckdb 1.5.5, pandas 3.0.5


## A1 — Размерность обучающего датасета

**Задание**

A1. Размерность
Сколько строк и колонок в обучающем датасете? Формат: строки; колонки. Обратите внимание: текстовое поле содержит переносы строк, наивное чтение файла даст неверный ответ.

Пример ответа: 123; 12

In [2]:
# Корректный подсчёт: CSV разбирается с учётом кавычек, поэтому переносы строк
# внутри Consumer.Complaint.Narrative не создают лишних записей.
n_rows = con.sql(f"SELECT count(*) FROM {TRAIN}").fetchone()[0]
columns = list(con.sql(f"SELECT * FROM {TRAIN} LIMIT 0").columns)
n_cols = len(columns)

print(f"Строк (записей):  {n_rows:,}".replace(",", " "))
print(f"Колонок:          {n_cols}")
print()
for i, name in enumerate(columns, 1):
    print(f"{i:>3}. {name}")

Строк (записей):  13 367 673
Колонок:          18

  1. complaint_id
  2. Date received
  3. date_sent_to_company
  4. Product
  5. sub_product
  6. Issue
  7. Sub.Issue
  8. Consumer.Complaint.Narrative
  9. company_public_response
 10. Company
 11. State
 12. ZIP code
 13. tags
 14. Submitted Via
 15. timely_response
 16. consumer_consent_region
 17. internal_priority_score
 18. relief_granted


In [3]:
# Демонстрация ловушки из условия: наивный подсчёт физических строк файла
# (аналог `wc -l`) сильно завышает число записей, потому что многострочный
# текст жалобы разрывается на несколько строк файла.
naive_newlines = 0
with open(TRAIN_CSV, "rb") as f:
    while chunk := f.read(1 << 24):
        naive_newlines += chunk.count(b"\n")

naive_rows = naive_newlines - 1  # минус строка заголовка

print(f"Наивно (переводы строк минус заголовок): {naive_rows:,}".replace(",", " "))
print(f"Корректно (парсинг CSV с кавычками):     {n_rows:,}".replace(",", " "))
print(f"Завышение при наивном чтении:            {naive_rows / n_rows:.2f}x")

Наивно (переводы строк минус заголовок): 23 307 172
Корректно (парсинг CSV с кавычками):     13 367 673
Завышение при наивном чтении:            1.74x


In [4]:
# Независимая проверка другим парсером (pyarrow, newlines_in_values=True):
# число записей и колонок не должно зависеть от движка чтения.
import pyarrow.csv as pv

reader = pv.open_csv(
    TRAIN_CSV,
    read_options=pv.ReadOptions(block_size=1 << 26),
    parse_options=pv.ParseOptions(newlines_in_values=True),
)
arrow_rows, arrow_cols = 0, None
for batch in reader:
    arrow_rows += batch.num_rows
    arrow_cols = batch.num_columns

print(f"pyarrow: строк = {arrow_rows:,}".replace(",", " ") + f", колонок = {arrow_cols}")
print(f"duckdb : строк = {n_rows:,}".replace(",", " ") + f", колонок = {n_cols}")
assert (arrow_rows, arrow_cols) == (n_rows, n_cols), "Парсеры разошлись в размерности"
print("\nОтвет A1 →", f"{n_rows}; {n_cols}")

pyarrow: строк = 13 367 673, колонок = 18
duckdb : строк = 13 367 673, колонок = 18

Ответ A1 → 13367673; 18


**Ответ:** 13367673; 18

**Вывод:** обучающий датасет содержит 13 367 673 записи и 18 колонок (17 признаков плюс таргет
`relief_granted`). Наивный подсчёт физических строк файла даёт 23 307 172 — завышение в ~1.74 раза,
потому что поле `Consumer.Complaint.Narrative` содержит переносы строк внутри закавыченных значений.
Одна запись данных ≠ одна строка файла, поэтому весь дальнейший анализ идёт через CSV-парсер,
учитывающий кавычки. Дополнительно зафиксировано: параллельный CSV-ридер DuckDB на этом файле
неприменим, чтение выполняется в режиме `parallel=false`; результат подтверждён вторым независимым
парсером (pyarrow).

## A2 — Пропуски: топ-3 колонки по доле пропусков

**Задание**

A2. Пропуски
Назовите топ-3 колонки по доле пропусков, по убыванию. Формат: col1, col2, col3 — имена колонок ровно как в файле.

Пример ответа: column_a, column_b, column_c

Данные грязные, поэтому пропуск в этом файле закодирован двумя способами, и оба нужно учесть:

1. **настоящий пропуск** — пустое поле в CSV (`NULL` после парсинга) или строка из одних пробелов;
2. **строка-заглушка** вместо значения — `-`, `.`, `null`, `NA`, `N/A`, `unknown` и подобные: формально это
   непустой текст, но информации в нём нет.

Ниже сначала показано, что заглушки действительно присутствуют, затем доли пропусков считаются по всем 18
колонкам в двух вариантах: «строго» (только `NULL` / пустая строка) и «широко» (плюс строки-заглушки).

In [5]:
# Доказательство, что кроме настоящих NULL есть строки-заглушки.
# Смотрим на короткие значения в колонках, где ожидаются пропуски (быстрая выборка сверху файла,
# нужна только для демонстрации самих токенов — итоговые доли считаются по всему датасету ниже).
con.execute(f"CREATE OR REPLACE TEMP TABLE a2_head AS SELECT * FROM {TRAIN} LIMIT 500000")

for col in ("tags", "company_public_response", "Sub.Issue"):
    df_tok = con.sql(f"""
        SELECT coalesce("{col}", '<NULL>') AS value, count(*) AS n
        FROM a2_head
        WHERE "{col}" IS NULL OR length(trim("{col}")) <= 8
        GROUP BY 1
        ORDER BY n DESC
        LIMIT 8
    """).df()
    print(f"--- {col} (короткие значения и NULL, первые 500k строк) ---")
    print(df_tok.to_string(index=False))
    print()

--- tags (короткие значения и NULL, первые 500k строк) ---
  value      n
 <NULL> 234104
      -  41455
unknown  41385
      .  41167
    N/A  41100
   null  40922
     NA  40814

--- company_public_response (короткие значения и NULL, первые 500k строк) ---
  value     n
 <NULL> 99373
      - 17676
   null 17646
      . 17523
    N/A 17471
unknown 17466
     NA 17425

--- Sub.Issue (короткие значения и NULL, первые 500k строк) ---
  value    n
 <NULL> 5766
   null 1076
      - 1032
unknown 1031
    N/A 1018
      . 1005
     NA  991



In [6]:
# Доли пропусков по всем 18 колонкам за один проход по полному CSV (~7.2 ГиБ, режим parallel=false,
# порядка 2-3 минут). Считаем сразу два определения пропуска, чтобы ответ не зависел от трактовки.

MISSING_TOKENS = [
    "", "-", "--", ".", "null", "na", "n/a", "n.a.", "nan", "none", "unknown", "missing", "?",
]
_tokens_sql = ", ".join(f"'{t}'" for t in MISSING_TOKENS)

_exprs = ["count(*) AS n_rows"]
for i, col in enumerate(columns):
    q = f'"{col}"'
    # строго: настоящий NULL или строка только из пробельных символов
    _exprs.append(f"sum(CASE WHEN {q} IS NULL OR trim({q}) = '' THEN 1 ELSE 0 END) AS strict_{i}")
    # широко: плюс строки-заглушки (регистр и обрамляющие пробелы не важны)
    _exprs.append(
        f"sum(CASE WHEN {q} IS NULL OR lower(trim({q})) IN ({_tokens_sql}) "
        f"THEN 1 ELSE 0 END) AS wide_{i}"
    )

_row = con.sql(f"SELECT {', '.join(_exprs)} FROM {TRAIN}").fetchone()

n_rows_a2 = _row[0]
assert n_rows_a2 == n_rows, "Число строк разошлось с A1"

missing = pd.DataFrame(
    {
        "column": columns,
        "n_strict": [_row[1 + 2 * i] for i in range(len(columns))],
        "n_wide": [_row[2 + 2 * i] for i in range(len(columns))],
    }
)
missing["share_strict"] = missing["n_strict"] / n_rows_a2
missing["share_wide"] = missing["n_wide"] / n_rows_a2

missing_sorted = missing.sort_values("share_wide", ascending=False).reset_index(drop=True)
print(f"Всего строк: {n_rows_a2:,}".replace(",", " "))
print("\nДоли пропусков по всем колонкам (сортировка по «широкому» определению):\n")
print(
    missing_sorted.assign(
        share_strict=lambda d: (d["share_strict"] * 100).round(3).astype(str) + "%",
        share_wide=lambda d: (d["share_wide"] * 100).round(3).astype(str) + "%",
    ).to_string(index=False)
)

Всего строк: 13 367 673

Доли пропусков по всем колонкам (сортировка по «широкому» определению):

                      column  n_strict   n_wide share_strict share_wide
     consumer_consent_region  13367673 13367673       100.0%     100.0%
                        tags   6241852 12856261      46.694%    96.174%
Consumer.Complaint.Narrative  10153017 10153017      75.952%    75.952%
     company_public_response   2665066  5485723      19.937%    41.037%
                   Sub.Issue    154987   319541       1.159%      2.39%
                       State     13956    28466       0.104%     0.213%
                    ZIP code       147      294       0.001%     0.002%
                 sub_product        36       86         0.0%     0.001%
                       Issue         7        7         0.0%       0.0%
                complaint_id         0        0         0.0%       0.0%
               Date received         0        0         0.0%       0.0%
                     Company         0

In [7]:
# Топ-3 по каждому определению пропуска.
top3_strict = missing.sort_values("share_strict", ascending=False).head(3)
top3_wide = missing.sort_values("share_wide", ascending=False).head(3)

print("Топ-3 по строгому определению (только NULL / пустая строка):")
for _, r in top3_strict.iterrows():
    print(f"  {r['column']:<30} {r['share_strict']:.4%}  ({r['n_strict']:,})".replace(",", " "))

print("\nТоп-3 по широкому определению (NULL + строки-заглушки):")
for _, r in top3_wide.iterrows():
    print(f"  {r['column']:<30} {r['share_wide']:.4%}  ({r['n_wide']:,})".replace(",", " "))

names_strict = list(top3_strict["column"])
names_wide = list(top3_wide["column"])

print(f"\nСостав тройки одинаков:  {set(names_strict) == set(names_wide)}")
print(f"Порядок совпадает:       {names_strict == names_wide}")

# Причина расхождения порядка — колонка tags: в ней много не настоящих NULL, а строк-заглушек.
tags_row = missing.loc[missing["column"] == "tags"].iloc[0]
print(
    f"\ntags: NULL/пусто = {tags_row['share_strict']:.4%}, "
    f"с заглушками = {tags_row['share_wide']:.4%}; "
    f"из них заглушки = {(tags_row['n_wide'] - tags_row['n_strict']):,}".replace(",", " ")
)
print("Реальные значения tags:")
print(
    con.sql("""
        SELECT tags AS value, count(*) AS n
        FROM a2_head
        WHERE tags IS NOT NULL
          AND lower(trim(tags)) NOT IN ('', '-', '.', 'null', 'na', 'n/a', 'unknown')
        GROUP BY 1 ORDER BY n DESC
    """).df().to_string(index=False)
)

# Ответ даётся по широкому определению: заглушки '-', '.', 'null', 'NA', 'N/A', 'unknown'
# в колонке tags не являются категориями (реальные категории — Older American / Servicemember),
# то есть информации в них нет и это фактические пропуски.
# Имена колонок берутся ровно как в заголовке файла, без переименования.
answer_a2 = ", ".join(names_wide)
print("\nОтвет A2 →", answer_a2)
print("Альтернативный порядок, если считать только настоящие NULL →", ", ".join(names_strict))

Топ-3 по строгому определению (только NULL / пустая строка):
  consumer_consent_region        100.0000%  (13 367 673)
  Consumer.Complaint.Narrative   75.9520%  (10 153 017)
  tags                           46.6936%  (6 241 852)

Топ-3 по широкому определению (NULL + строки-заглушки):
  consumer_consent_region        100.0000%  (13 367 673)
  tags                           96.1743%  (12 856 261)
  Consumer.Complaint.Narrative   75.9520%  (10 153 017)

Состав тройки одинаков:  True
Порядок совпадает:       False

tags: NULL/пусто = 46.6936%  с заглушками = 96.1743%; из них заглушки = 6 614 409
Реальные значения tags:


                        value     n
               Older American 13124
Older American, Servicemember  4591
                Servicemember  1338

Ответ A2 → consumer_consent_region, tags, Consumer.Complaint.Narrative
Альтернативный порядок, если считать только настоящие NULL → consumer_consent_region, Consumer.Complaint.Narrative, tags


**Ответ:** consumer_consent_region, tags, Consumer.Complaint.Narrative

**Вывод:**

| колонка | только NULL / пусто | NULL + строки-заглушки |
|---|---|---|
| `consumer_consent_region` | 100.000% | 100.000% |
| `tags` | 46.694% | **96.174%** |
| `Consumer.Complaint.Narrative` | 75.952% | 75.952% |
| `company_public_response` | 19.937% | 41.037% |

Состав тройки не зависит от трактовки пропуска, но **порядок 2-го и 3-го места зависит**, и это важно
проговорить явно:

- `consumer_consent_region` пуста полностью — 13 367 673 из 13 367 673 (100%), первое место при любой
  трактовке. Признак бесполезен для модели.
- `tags` содержит настоящих `NULL` 46.694%, но остальные «значения» — заглушки `-`, `.`, `null`, `NA`,
  `N/A`, `unknown`; содержательных категорий всего две (`Older American`, `Servicemember`). Считая
  заглушки пропусками, доля пропусков — 96.174%, и колонка выходит на 2-е место.
- `Consumer.Complaint.Narrative` пропущена в 75.952% случаев (10 153 017 строк) — здесь оба определения
  совпадают, заглушек нет. Это ограничивает применимость текстового признака: текст есть лишь у четверти
  жалоб, поэтому NLP-пайплайн должен работать при отсутствующем тексте.

Ответ дан по «широкому» определению, потому что строка `unknown` или `-` — это закодированный пропуск, а не
значение признака. Если считать пропусками только настоящие `NULL`/пустые поля, порядок будет
`consumer_consent_region, Consumer.Complaint.Narrative, tags`.

Ещё одно наблюдение для дальнейшей работы: `company_public_response` (41.037% с учётом заглушек) в тройку
не входит, но это поле формируется **после** ответа компании на жалобу, то есть при прогнозировании его
может не быть — риск утечки нужно проверить отдельно.